In [1]:
from __future__ import annotations
import argparse
import json
import math
import os
import random
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple


import numpy as np
from PIL import Image, ImageDraw


import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

In [2]:
# -----------------------------
# Config: canonical parts (13)
# -----------------------------
CANONICAL_PARTS: List[str] = [
"head", "body", "wing", "tail", "leg", "foot", "hand", "eye", "ear", "mouth", "wheel", "handle", "misc"
]
PART_TO_IDX: Dict[str, int] = {p: i for i, p in enumerate(CANONICAL_PARTS)}


# Raw → canonical mapping (add your synonyms here)
PART_SYNONYMS: Dict[str, str] = {
# heads / faces
"face": "head", "skull": "head", "nose": "head", "beak": "head", "snout": "head",
# bodies / torsos
"torso": "body", "abdomen": "body", "chest": "body", "trunk": "body",
# limbs
"paw": "foot", "feet": "foot", "hoof": "foot", "flipper": "hand", "arm": "hand",
"hind_leg": "leg", "front_leg": "leg",
# vehicle-ish
"tyre": "wheel", "tire": "wheel", "handlebar": "handle", "grip": "handle",
}


# Fallback keys if your JSON uses different field names
KEY_FALLBACKS = {
"part_name": ["part_name", "part", "partCategory", "part_category", "label_part", "partName"],
"category_name": ["category_name", "supercategory", "object", "object_name", "category", "superCategory"],
"category_id": ["category_id", "object_category_id", "obj_cat", "categoryId"],
"bbox": ["bbox", "bBox", "box"],
"segmentation": ["segmentation", "seg", "polygon"],
"file_name": ["file_name", "path", "image", "name", "fileName"],
}

In [3]:
# Utility
# -----------------------------

def seed_all(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def find_key(d: Dict[str, Any], candidates: List[str], default=None):
    for k in candidates:
        if k in d:
            return d[k]
    return default


def canonical_part(raw: str) -> str:
    if raw is None:
        return "misc"
    r = raw.strip().lower()
    r = PART_SYNONYMS.get(r, r)
    return r if r in PART_TO_IDX else "misc"


def box_from_polygon(poly: List[float]) -> List[float]:
    xs = np.array(poly[0::2])
    ys = np.array(poly[1::2])
    x0, x1 = xs.min(), xs.max()
    y0, y1 = ys.min(), ys.max()
    return [float(x0), float(y0), float(x1 - x0), float(y1 - y0)]


def pil_crop_from_ann(img: Image.Image, ann: Dict[str, Any]) -> Optional[Image.Image]:
    """Crop **RGB** patch only. Prefer bbox; if only polygon segmentation exists, use its tight bbox.
    """
    bbox = find_key(ann, KEY_FALLBACKS["bbox"], None)
    seg = find_key(ann, KEY_FALLBACKS["segmentation"], None)

    # Prefer bbox if provided; else fall back to polygon bbox
    if bbox is not None:
        bx = bbox
    elif seg and isinstance(seg, list) and len(seg) > 0:
        poly = seg[0] if isinstance(seg[0], list) else seg
        bx = box_from_polygon(poly)
    else:
        return None

    # Support dict-style bbox as well as [x,y,w,h]
    if isinstance(bx, dict):
        x = float(bx.get("x", bx.get("left", 0)))
        y = float(bx.get("y", bx.get("top", 0)))
        w = float(bx.get("w", bx.get("width", 0)))
        h = float(bx.get("h", bx.get("height", 0)))
    else:
        x, y, w, h = map(float, bx)

    x0, y0, x1, y1 = int(max(0, x)), int(max(0, y)), int(min(img.width, x + w)), int(min(img.height, y + h))
    if x1 <= x0 or y1 <= y0:
        return None
    return img.crop((x0, y0, x1, y1))

In [4]:
# Data structures
# -----------------------------
@dataclass
class PartExample:
    img_path: str
    part_canonical: str
    object_super: str  # used as "cluster" label within a detector
    bbox_or_seg_ann: Dict[str, Any]
    image_id: int


class PartCropDataset(Dataset):
    def __init__(
        self,
        root: str,
        json_path: str,
        images_root: str,
        transform: Optional[T.Compose] = None,
        min_size: int = 12,
    ):
        super().__init__()
        self.root = root
        self.transform = transform or T.Compose([
            T.Resize((128, 128)),
            T.ToTensor(),
            T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ])

        with open(os.path.join(root, json_path), "r") as f:
            j = json.load(f)

        # Build category map if present (COCO-style)
        self.cat_map: Dict[int, Dict[str, Any]] = {}
        for c in j.get("categories", []):
            if "id" in c:
                self.cat_map[int(c["id"])] = c

        # Build image map (be flexible with file_name formatting)
        images = j.get("images", [])
        self.image_map = {}
        for im in images:
            fn = find_key(im, KEY_FALLBACKS["file_name"], None)
            if fn is None:
                continue
            # Choose an existing path among candidates
            candidates = []
            if os.path.isabs(fn):
                candidates.append(fn)
            if images_root:
                candidates.append(os.path.join(root, images_root, fn))
            candidates.append(os.path.join(root, fn))
            full = None
            for pth in candidates:
                if os.path.exists(pth):
                    full = pth
                    break
            # Fall back to join even if not existing (for later existence checks)
            if full is None:
                full = candidates[0] if candidates else os.path.join(root, fn)
            im_id = im.get("id", im.get("image_id", len(self.image_map)))
            self.image_map[im_id] = full

        anns = j.get("annotations", [])
        self.items: List[PartExample] = []
        for a in anns:
            img_id = a.get("image_id", a.get("id", None))
            if img_id is None or img_id not in self.image_map:
                continue
            img_path = self.image_map[img_id]
            if not os.path.exists(img_path):
                continue

            # Determine object supercategory/name and category record
            obj = find_key(a, KEY_FALLBACKS["category_name"], None)
            cat = None
            cat_id = find_key(a, KEY_FALLBACKS["category_id"], None)
            if obj is None and cat_id is not None and len(self.cat_map) > 0:
                try:
                    cat = self.cat_map[int(cat_id)]
                    obj = cat.get("supercategory") or cat.get("name") or str(cat_id)
                except Exception:
                    obj = str(cat_id)
            if obj is None:
                continue

            # Infer part: prefer explicit part_name, else derive from category name like 'Bird Wing' → 'wing'
            raw_part = find_key(a, KEY_FALLBACKS["part_name"], None)
            if raw_part is None:
                if cat is None and cat_id is not None and int(cat_id) in self.cat_map:
                    cat = self.cat_map[int(cat_id)]
                cat_name = cat.get("name") if cat is not None else None
                part = derive_part_from_category_name(cat_name)
            else:
                part = canonical_part(raw_part)

            # quick reject if bbox is tiny (if present)
            bbox = find_key(a, KEY_FALLBACKS["bbox"], None)
            if bbox is not None:
                try:
                    _, _, w, h = bbox
                except Exception:
                    w = float(bbox.get("width", 0)); h = float(bbox.get("height", 0))
                if float(w) < min_size or float(h) < min_size:
                    continue

            self.items.append(PartExample(
                img_path=img_path,
                part_canonical=part,
                object_super=str(obj),
                bbox_or_seg_ann=a,
                image_id=img_id,
            ))

        # Build per-detector cluster dictionaries (on-the-fly, from this split)
        self.detector_clusters: Dict[str, Dict[str, int]] = {}
        for p in CANONICAL_PARTS:
            objs = sorted({it.object_super for it in self.items if it.part_canonical == p})
            self.detector_clusters[p] = {o: i for i, o in enumerate(objs)}

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, idx: int):
        it = self.items[idx]
        img = Image.open(it.img_path).convert("RGB")
        crop = pil_crop_from_ann(img, it.bbox_or_seg_ann)
        if crop is None:
            # on bad crop, return a black image + label to keep loader simple
            crop = Image.new("RGB", (128, 128), (0, 0, 0))
        x = self.transform(crop)
        det_idx = PART_TO_IDX[it.part_canonical]
        cluster_idx = self.detector_clusters[it.part_canonical].get(it.object_super, -1)
        return x, det_idx, cluster_idx, it.image_id


In [5]:
# Models: MiniResNet8 (very small)
# -----------------------------
# MiniResNet8 (very small)
# -----------------------------
class BasicBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, stride: int = 1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.shortcut = None
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.shortcut is not None:
            x = self.shortcut(x)
        out = F.relu(out + x)
        return out


class MiniResNet8(nn.Module):
    """A tiny ResNet-ish backbone with ~8 conv layers total."""
    def __init__(self, in_ch: int = 3, width: int = 32):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, width, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(width),
            nn.ReLU(inplace=True),
        )
        self.layer1 = BasicBlock(width, width, stride=1)
        self.layer2 = BasicBlock(width, width * 2, stride=2)
        self.layer3 = BasicBlock(width * 2, width * 4, stride=2)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.out_ch = width * 4

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.pool(x).flatten(1)
        return x  # (B, out_ch)


class PartDetectorNet(nn.Module):
    def __init__(self, emb_dim: int, num_clusters: int, width: int = 32):
        super().__init__()
        self.backbone = MiniResNet8(width=width)
        self.proj = nn.Linear(self.backbone.out_ch, emb_dim)
        self.cls = nn.Linear(emb_dim, num_clusters) if num_clusters > 0 else None

    def forward(self, x):
        h = self.backbone(x)
        z = F.normalize(self.proj(h), dim=-1)
        logits = self.cls(z) if self.cls is not None else None
        return z, logits

In [6]:
# Train utilities
# -----------------------------

def make_dataloaders(ds: PartCropDataset, batch_size: int = 128, num_workers: int = 2):
    return DataLoader(ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True, drop_last=False)


def train_detectors(
    root: str,
    train_json: str,
    images_root: str,
    outdir: str,
    emb_dim: int = 128,
    width: int = 32,
    epochs: int = 10,
    batch_size: int = 128,
    lr: float = 1e-3,
    device: str = "cuda" if torch.cuda.is_available() else "cpu",
) -> None:
    os.makedirs(outdir, exist_ok=True)

    train_ds = PartCropDataset(root, train_json, images_root)
    loader = make_dataloaders(train_ds, batch_size=batch_size)

    # Build a detector model per canonical part
    detectors: Dict[str, PartDetectorNet] = {}
    optimizers: Dict[str, torch.optim.Optimizer] = {}
    criterions: Dict[str, nn.Module] = {}
    cluster_priors: Dict[str, List[float]] = {}

    for part in CANONICAL_PARTS:
        num_clu = len(train_ds.detector_clusters[part])
        model = PartDetectorNet(emb_dim=emb_dim, num_clusters=num_clu, width=width).to(device)
        detectors[part] = model
        if num_clu > 0:
            optimizers[part] = torch.optim.AdamW(model.parameters(), lr=lr)
            criterions[part] = nn.CrossEntropyLoss()
            # simple frequency prior
            cluster_priors[part] = [1e-8] * num_clu
        else:
            cluster_priors[part] = []

    # Count priors
    for _, det_idx, cluster_idx, _ in loader:
        for d, c in zip(det_idx.tolist(), cluster_idx.tolist()):
            part = CANONICAL_PARTS[d]
            if c >= 0 and len(cluster_priors[part]) > 0:
                cluster_priors[part][c] += 1.0
    for p, arr in cluster_priors.items():
        s = sum(arr) if arr else 1.0
        if s > 0:
            cluster_priors[p] = [x / s for x in arr]

    # Train
    for epoch in range(1, epochs + 1):
        for part in CANONICAL_PARTS:
            if len(train_ds.detector_clusters[part]) == 0:
                continue
            detectors[part].train()

        for xb, det_idx, cluster_idx, _ in loader:
            xb = xb.to(device)
            det_idx = det_idx.to(device)
            cluster_idx = cluster_idx.to(device)

            # Split batch by detector and update each independently
            for part_i, part in enumerate(CANONICAL_PARTS):
                mask = (det_idx == part_i)
                if not mask.any():
                    continue
                xb_p = xb[mask]
                y_p = cluster_idx[mask]
                # Ignore -1 cluster labels (if any)
                valid = (y_p >= 0)
                if not valid.any():
                    continue
                xb_p = xb_p[valid]
                y_p = y_p[valid]

                model = detectors[part]
                model.train()
                opt = optimizers[part]
                crit = criterions[part]

                z, logits = model(xb_p)
                loss = crit(logits, y_p)
                opt.zero_grad()
                loss.backward()
                opt.step()

        print(f"[Epoch {epoch}/{epochs}] detectors updated.")

    # Save all detectors + label maps + priors
    for part in CANONICAL_PARTS:
        model = detectors[part]
        torch.save({
            "state_dict": model.state_dict(),
            "emb_dim": emb_dim,
            "width": width,
            "clusters": train_ds.detector_clusters[part],  # dict: object_super -> idx
            "priors": cluster_priors[part],
        }, os.path.join(outdir, f"det_{part}.pt"))

    # Also dump a summary mapping
    with open(os.path.join(outdir, "clusters_summary.json"), "w") as f:
        json.dump({p: m for p, m in train_ds.detector_clusters.items()}, f, indent=2)
    print(f"Saved 13 detectors and cluster maps to: {outdir}")


In [7]:
# Inference / Fusion
# -----------------------------
class LoadedDetector:
    def __init__(self, part: str, ckpt: Dict[str, Any], device: str):
        self.part = part
        self.emb_dim = ckpt["emb_dim"]
        self.width = ckpt["width"]
        self.clusters = ckpt["clusters"]  # dict: obj name -> id
        self.inv_clusters = {v: k for k, v in self.clusters.items()}
        self.priors = torch.tensor(ckpt.get("priors", []), dtype=torch.float32, device=device)
        self.model = PartDetectorNet(self.emb_dim, len(self.clusters), self.width).to(device)
        self.model.load_state_dict(ckpt["state_dict"])  # type: ignore
        self.model.eval()

    @torch.no_grad()
    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        z, logits = self.model(x)
        probs = logits.softmax(dim=-1) if logits is not None else None
        return z, probs


class PartEnsemble:
    def __init__(self, detectors_dir: str, device: str):
        self.device = device
        self.detectors: List[LoadedDetector] = []
        for p in CANONICAL_PARTS:
            path = os.path.join(detectors_dir, f"det_{p}.pt")
            if not os.path.exists(path):
                raise FileNotFoundError(f"Missing detector checkpoint: {path}")
            ckpt = torch.load(path, map_location=device)
            self.detectors.append(LoadedDetector(p, ckpt, device))
        self.emb_dim = self.detectors[0].emb_dim

    @torch.no_grad()
    def assign_part(self, x: torch.Tensor) -> Tuple[int, int, torch.Tensor, float]:
        """
        x: (1, 3, H, W) normalized crop
        Returns: (detector_idx, cluster_idx, embedding, score)
        score = weighted probability = sum_c P(c|x,det) * prior(c|det)
        """
        best_det = -1
        best_cluster = -1
        best_z = None
        best_score = -1.0
        for i, det in enumerate(self.detectors):
            z, probs = det.forward(x)
            if probs is None or probs.numel() == 0:
                continue
            score = (probs * det.priors.unsqueeze(0)).sum(dim=-1)  # (B,)
            s = float(score.item())
            if s > best_score:
                best_score = s
                best_det = i
                best_cluster = int(probs.argmax(dim=-1).item())
                best_z = z
        if best_z is None:
            best_z = torch.zeros((1, self.emb_dim), device=self.device)
        return best_det, best_cluster, best_z.squeeze(0), best_score

    @torch.no_grad()
    def fuse_image_parts(
        self,
        ds: PartCropDataset,
        image_id: int,
        transform: Optional[T.Compose] = None,
    ) -> Tuple[torch.Tensor, Optional[str]]:
        """
        Build the 13*emb vector for a given image by assigning each part crop to exactly one detector slot.
        Returns (fused_vector, object_supercategory_if_all_parts_agree_or_None)
        """
        transform = transform or T.Compose([
            T.Resize((128, 128)), T.ToTensor(),
            T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ])
        device = self.device
        # collect part items for this image
        items = [it for it in ds.items if it.image_id == image_id]
        fused = torch.zeros((len(CANONICAL_PARTS) * self.emb_dim,), device=device)
        obj_names = []
        for it in items:
            img = Image.open(it.img_path).convert("RGB")
            crop = pil_crop_from_ann(img, it.bbox_or_seg_ann)
            if crop is None:
                continue
            x = transform(crop).unsqueeze(0).to(device)
            det_idx, clu_idx, z, score = self.assign_part(x)
            if det_idx >= 0:
                s = det_idx * self.emb_dim
                fused[s:s + self.emb_dim] = z
                obj_names.append(it.object_super)
        # naive consensus of object name
        maj = None
        if len(obj_names) > 0:
            # majority vote
            counts = {}
            for o in obj_names:
                counts[o] = counts.get(o, 0) + 1
            maj = max(counts.items(), key=lambda kv: kv[1])[0]
        return fused, maj

In [8]:
# Final linear head (on fused vectors)
# -----------------------------
class FinalHead(nn.Module):
    def __init__(self, in_dim: int, num_classes: int):
        super().__init__()
        self.fc = nn.Linear(in_dim, num_classes)
    def forward(self, x):
        return self.fc(x)


def build_image_level_dataset(
    ds: PartCropDataset,
    ensemble: PartEnsemble,
    label_from_majority: bool = True,
) -> Tuple[torch.Tensor, torch.Tensor, Dict[int, str], Dict[str, int]]:
    """
    Returns: (X, y, id_to_obj, obj_to_id)
      X: (N_images, 13*emb)
      y: (N_images,) long labels (object supercategory)
    """
    device = ensemble.device
    # collect distinct image_ids
    image_ids = sorted({it.image_id for it in ds.items})
    obj_names = sorted({it.object_super for it in ds.items})
    obj_to_id = {o: i for i, o in enumerate(obj_names)}
    id_to_obj = {i: o for o, i in obj_to_id.items()}

    X_list = []
    y_list = []
    for im_id in image_ids:
        fused, maj = ensemble.fuse_image_parts(ds, im_id)
        X_list.append(fused.cpu().numpy())
        # Choose label
        if label_from_majority and maj is not None:
            y_list.append(obj_to_id[maj])
        else:
            # fallback: pick the most frequent object name in this image's parts
            objs = [it.object_super for it in ds.items if it.image_id == im_id]
            counts = {}
            for o in objs:
                counts[o] = counts.get(o, 0) + 1
            if len(counts) == 0:
                continue
            maj = max(counts.items(), key=lambda kv: kv[1])[0]
            y_list.append(obj_to_id[maj])

    X = torch.tensor(np.stack(X_list), dtype=torch.float32)
    y = torch.tensor(np.array(y_list), dtype=torch.long)
    return X, y, id_to_obj, obj_to_id


def train_final_head(
    X_train: torch.Tensor, y_train: torch.Tensor,
    X_val: Optional[torch.Tensor] = None, y_val: Optional[torch.Tensor] = None,
    epochs: int = 20, lr: float = 5e-3, weight_decay: float = 0.0,
    device: str = "cuda" if torch.cuda.is_available() else "cpu",
) -> FinalHead:
    in_dim = X_train.shape[1]
    num_classes = int(y_train.max().item() + 1)
    model = FinalHead(in_dim, num_classes).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    crit = nn.CrossEntropyLoss()

    X_train = X_train.to(device); y_train = y_train.to(device)
    if X_val is not None:
        X_val = X_val.to(device); y_val = y_val.to(device)

    for ep in range(1, epochs + 1):
        model.train()
        logits = model(X_train)
        loss = crit(logits, y_train)
        opt.zero_grad(); loss.backward(); opt.step()
        msg = f"[Head] Epoch {ep}/{epochs} loss={loss.item():.4f}"
        if X_val is not None:
            model.eval()
            with torch.no_grad():
                acc = (model(X_val).argmax(-1) == y_val).float().mean().item()
            msg += f" val_acc={acc:.3f}"
        print(msg)
    return model

In [9]:
# CLI
# -----------------------------

def parse_args():
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", type=str, required=True, help="Path to PartImageNet root (sample/PartImageNet)")
    ap.add_argument("--train-ann", type=str, required=True, help="Relative JSON for train annotations, e.g., annotations/train/train.json")
    ap.add_argument("--images-root", type=str, required=True, help="Relative path under root to images (e.g., images/train or images)")
    ap.add_argument("--test-ann", type=str, default=None, help="Relative JSON for test annotations (optional)")

    ap.add_argument("--stage", type=str, required=True, choices=["train_detectors", "train_head", "dump_vectors"],
                    help="What to run")
    ap.add_argument("--detectors", type=str, default="checkpoints", help="Dir with trained det_*.pt files or output dir to save")
    ap.add_argument("--outdir", type=str, default="checkpoints", help="Where to save detectors if training")

    ap.add_argument("--epochs", type=int, default=10)
    ap.add_argument("--batch-size", type=int, default=128)
    ap.add_argument("--lr", type=float, default=1e-3)
    ap.add_argument("--emb-dim", type=int, default=128)
    ap.add_argument("--width", type=int, default=32)
    ap.add_argument("--dump-npy", type=str, default=None, help="When stage=dump_vectors, file to save .npy array")

    return ap.parse_args()

In [10]:
from part_detectors_pipeline import (
    PartCropDataset, PartEnsemble, train_detectors,
    build_image_level_dataset, train_final_head
)
import torch, os

ROOT = "PartImageNet_Seg/PartImageNet"

# 1) Train detectors
train_detectors(
    root=ROOT,
    train_json="annotations/train/train.json",
    images_root="image/train",        # or "image" if file_name already has train/
    outdir="checkpoints",
    emb_dim=128, width=32,
    epochs=3, batch_size=128, lr=1e-3,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

# 2) Fuse + head
ds_tr = PartCropDataset(ROOT, "annotations/train/train.json", "image/train")
ens   = PartEnsemble("checkpoints", device="cuda" if torch.cuda.is_available() else "cpu")
Xtr, ytr, *_ = build_image_level_dataset(ds_tr, ens)

ds_te = PartCropDataset(ROOT, "annotations/test/test.json", "image/test/jpegs")
Xte, yte, *_ = build_image_level_dataset(ds_te, ens)

head = train_final_head(Xtr, ytr, Xte, yte, epochs=10, lr=5e-3)
os.makedirs("checkpoints", exist_ok=True)
torch.save({"state_dict": head.state_dict(), "in_dim": Xtr.shape[1]}, "checkpoints/final_head.pt")


ValueError: num_samples should be a positive integer value, but got num_samples=0